# Lab Day 2 — DeepWeeds: Backbone, Training Recipe & Inference (Kaggle Edition)

- **Học viên:** Trần Thanh Thái
- **MSSV:** 2A202602454
- **Bài nộp:** Track 4 — Day 2: Phân loại cỏ dại DeepWeeds
- **Môi trường thực thi tối ưu:** **Kaggle Notebooks (GPU Tesla T4)**

---

### 🛡️ Tối Ưu Hóa Chống Lỗi Vượt 500 Files Của Kaggle:
* Kaggle giới hạn thư mục output (`/kaggle/working`) chỉ được chứa **tối đa 500 files**.
* Do dataset DeepWeeds có tới **17.509 ảnh**, notebook này tự động giải nén ảnh vào thư mục tạm **`/tmp/data`** (hoặc `/kaggle/temp/data`). 
* Thư mục output `/kaggle/working` sẽ chỉ chứa các file báo cáo và dự đoán (`results.xlsx`, `submission_outputs.zip`, `predictions/`, `curves/`), tổng cộng chỉ khoảng **20–30 files**, hoàn toàn an toàn và không bao giờ bị báo lỗi `Exceeded file count limit: 500 files`!

## 0. Cài đặt thư viện & Kiểm tra Phần cứng GPU

In [ ]:
# 1. Tắt các cảnh báo không cần thiết và cài đặt thư viện
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

!pip -q install timm openpyxl matplotlib scikit-learn scipy

import os, sys, platform, hashlib, time, json, shutil, subprocess, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import timm

# 2. Kiểm tra thông số GPU trên Kaggle
print("=" * 60)
print("THÔNG TIN MÔI TRƯỜNG KAGGLE:")
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Model:", torch.cuda.get_device_name(0))
    print("GPU VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("⚠️ CẢNH BÁO: GPU chưa được kích hoạt! Hãy vào Settings (cột phải) > Accelerator > GPU T4 x 1.")
print("=" * 60)

## 0.1 Đồng bộ mã nguồn bài lab từ GitHub
Tự động đồng bộ các module kỹ thuật: `dataset.py`, `model.py`, `losses.py`, `train.py`, `inference.py`, `benchmark.py`, `eval.py`, `export_results.py`.

In [ ]:
# Đồng bộ mã nguồn bài lab
REPO_URL = "https://github.com/tranthai239/K4-DAY02-TranThanhThai-2A202602454-MSSV.git"
REPO_NAME = "K4-DAY02-TranThanhThai-2A202602454-MSSV"
RAW_BASE = "https://raw.githubusercontent.com/tranthai239/K4-DAY02-TranThanhThai-2A202602454-MSSV/main"

# 1. Ưu tiên git clone
if not os.path.exists("dataset.py"):
    if not os.path.exists(REPO_NAME):
        print(f"Đang clone repo bài lab từ {REPO_URL}...")
        !git clone -q {REPO_URL}
    
    code_sub = f"{REPO_NAME}/submissions/2A202602454_TranThanhThai/code"
    if os.path.exists(code_sub):
        for f in os.listdir(code_sub):
            if f.endswith(".py") and not os.path.exists(f):
                shutil.copy(os.path.join(code_sub, f), f)
                
    if os.path.exists(f"{REPO_NAME}/eval.py") and not os.path.exists("eval.py"):
        shutil.copy(f"{REPO_NAME}/eval.py", "eval.py")

# 2. Cơ chế dự phòng tải trực tiếp qua curl
required_modules = [
    ("eval.py", f"{RAW_BASE}/eval.py"),
    ("dataset.py", f"{RAW_BASE}/submissions/2A202602454_TranThanhThai/code/dataset.py"),
    ("model.py", f"{RAW_BASE}/submissions/2A202602454_TranThanhThai/code/model.py"),
    ("losses.py", f"{RAW_BASE}/submissions/2A202602454_TranThanhThai/code/losses.py"),
    ("train.py", f"{RAW_BASE}/submissions/2A202602454_TranThanhThai/code/train.py"),
    ("inference.py", f"{RAW_BASE}/submissions/2A202602454_TranThanhThai/code/inference.py"),
    ("benchmark.py", f"{RAW_BASE}/submissions/2A202602454_TranThanhThai/code/benchmark.py"),
    ("export_results.py", f"{RAW_BASE}/submissions/2A202602454_TranThanhThai/code/export_results.py")
]

for filename, url in required_modules:
    if not os.path.exists(filename):
        print(f"Tải file dự phòng {filename}...")
        !curl -sL "{url}" -o "{filename}"

sys.path.insert(0, os.path.abspath("."))

import dataset
import model as model_utils
import losses as loss_utils
import inference as infer_utils
import benchmark as bench_utils
import train as train_module
import eval as ev
import export_results

print("✅ ĐÃ ĐỒNG BỘ VÀ IMPORT THÀNH CÔNG TOÀN BỘ CÁC MODULE!")

## 0.2 Tải và Chuẩn bị Dataset DeepWeeds (Tối ưu chống lỗi 500 files của Kaggle)
- Lưu trữ 17.509 ảnh trong thư mục tạm **`/tmp/data`** để **không tính vào giới hạn 500 files output** của Kaggle.
- Tự động nhận diện nếu bạn đã gắn dataset DeepWeeds từ Kaggle Input (`/kaggle/input/...`).
- Tự động dọn dẹp các file ảnh cũ trong `/kaggle/working` nếu có.

In [ ]:
# 1. Dọn dẹp thư mục ảnh cũ trong /kaggle/working để tránh vượt giới hạn 500 files
for old_dir in ["data/images", "data", "images"]:
    if os.path.exists(old_dir):
        print(f"Dọn dẹp thư mục '{old_dir}' khỏi /kaggle/working để không vượt quá 500 file output...")
        shutil.rmtree(old_dir, ignore_errors=True)
if os.path.exists("data/images.zip"):
    try:
        os.remove("data/images.zip")
    except Exception:
        pass

# 2. Đặt thư mục dữ liệu tại /tmp/data (hoặc /kaggle/temp/data)
if os.path.exists("/kaggle"):
    TEMP_ROOT = Path("/kaggle/temp/data") if os.path.exists("/kaggle/temp") else Path("/tmp/data")
else:
    TEMP_ROOT = Path("/tmp/data") if os.path.exists("/tmp") else Path("data")

TEMP_ROOT.mkdir(parents=True, exist_ok=True)
TEMP_IMAGES_DIR = TEMP_ROOT / "images"
TEMP_LABELS_DIR = TEMP_ROOT / "labels"
TEMP_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
TEMP_LABELS_DIR.mkdir(parents=True, exist_ok=True)

IMAGES_DIR = None
LABELS_DIR = None

# 3. Kiểm tra trong /kaggle/input (nếu đã thêm Kaggle Dataset)
kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    for p in kaggle_input.rglob("*.jpg"):
        IMAGES_DIR = str(p.parent)
        print(f"🎯 Phát hiện thư mục ảnh từ Kaggle Input: {IMAGES_DIR}")
        break
    for p in kaggle_input.rglob("train_subset0.csv"):
        LABELS_DIR = str(p.parent)
        print(f"🎯 Phát hiện thư mục nhãn từ Kaggle Input: {LABELS_DIR}")
        break

# 4. Nếu chưa có trên Kaggle Input, tải và giải nén vào thư mục tạm /tmp/data
if IMAGES_DIR is None:
    zip_path = TEMP_ROOT / "images.zip"
    if not zip_path.exists() or zip_path.stat().st_size < 400 * 1024 * 1024:
        print(f"Đang tải images.zip (~490MB) từ Zenodo vào {zip_path}...")
        !curl -L -s -o "{zip_path}" "https://zenodo.org/records/7939060/files/images.zip?download=1"

    # Kiểm tra MD5
    h = hashlib.md5()
    with open(zip_path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    md5_val = h.hexdigest()
    print(f"MD5 Checksum: {md5_val}")
    if md5_val == "b7b30f96d466fba86016aa5a26606e0f":
        print("-> MD5 trùng khớp 100% chuẩn Zenodo!")
    else:
        print(f"Lưu ý: MD5 {md5_val}")

    # Giải nén ảnh vào thư mục tạm
    if len(list(TEMP_IMAGES_DIR.glob("*.jpg"))) < 17000:
        print(f"Đang giải nén ảnh vào thư mục tạm {TEMP_ROOT}...")
        !unzip -q -o "{zip_path}" -d "{TEMP_ROOT}"
    
    if (TEMP_ROOT / "images").exists() and len(list((TEMP_ROOT / "images").glob("*.jpg"))) > 1000:
        IMAGES_DIR = str(TEMP_ROOT / "images")
    else:
        IMAGES_DIR = str(TEMP_ROOT)

if LABELS_DIR is None:
    BASE_URL = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
    for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
        dst = TEMP_LABELS_DIR / f"{name}.csv"
        if not dst.exists():
            !curl -sL "{BASE_URL}/{name}.csv" -o "{dst}"
    LABELS_DIR = str(TEMP_LABELS_DIR)

print(f"\n✅ ĐÃ SẴN SÀNG DỮ LIỆU TẠI THƯ MỤC AN TOÀN (Không gây lỗi 500 files output):")
print(f"- IMAGES_DIR = '{IMAGES_DIR}' ({len(os.listdir(IMAGES_DIR))} files)")
print(f"- LABELS_DIR = '{LABELS_DIR}'")

## 0.3 Cấu hình Chế độ Chạy (Execution Mode)
- `"FAST_VERIFY"`: Chạy kiểm tra nhanh 1-2 epochs mỗi cấu hình (~10-15 phút) để xác nhận code chạy trơn tru không lỗi.
- `"FULL_TRAIN"`: Chạy 10-12 epochs đầy đủ theo chuẩn GUIDE.md (khuyên dùng khi Commit Save & Run All).
- `"EVAL_ONLY"`: Bỏ qua train, sử dụng predictions có sẵn để chạy `eval.py` chấm điểm và xuất `results.xlsx`.

In [ ]:
# ==========================================
# CHỌN CHẾ ĐỘ CHẠY TẠI ĐÂY:
# ==========================================
MODE = "FAST_VERIFY"   # Đổi thành "FULL_TRAIN" khi muốn huấn luyện toàn diện 100%

if MODE == "FAST_VERIFY":
    EPOCHS_SCREEN = 1
    EPOCHS_FINAL = 2
    BATCH_SIZE = 32
    print(f"🔧 Chế độ FAST_VERIFY: Sàng lọc={EPOCHS_SCREEN} ep, Chung kết={EPOCHS_FINAL} ep, Batch size={BATCH_SIZE}")
elif MODE == "FULL_TRAIN":
    EPOCHS_SCREEN = 10
    EPOCHS_FINAL = 12
    BATCH_SIZE = 32
    print(f"🚀 Chế độ FULL_TRAIN: Sàng lọc={EPOCHS_SCREEN} ep, Chung kết={EPOCHS_FINAL} ep, Batch size={BATCH_SIZE}")
else:
    print(f"⚡ Chế độ EVAL_ONLY")

## Bước 0 — EDA & Kiểm tra Quy chuẩn Pipeline (Checklist slide p.59, README mục 2.1)
1. Kiểm tra phân tầng chia dữ liệu Fold 0: Train/Val/Test (60/20/20).
2. Kiểm tra giao giữa các tập rỗng, hợp đủ 17.509 ảnh.
3. Trực quan hóa phân bố nhãn (mất cân bằng lớp với `Negative` 52%).
4. Kiểm tra loss ban đầu $\approx -\ln(1/9) \approx 2.197$.
5. Kiểm tra overfit 1 mini-batch tới loss $\approx 0$.

In [ ]:
# 1. Đọc và kiểm tra chia dữ liệu Fold 0 (S1-S6)
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
stats = dataset.check_split(train_df, val_df, test_df, images_dir=IMAGES_DIR)

print("=== KẾT QUẢ KIỂM TRA CHIA DỮ LIỆU FOLD 0 ===")
print("Số lượng ảnh từng tập:", stats["n"])
print("Tỉ lệ (kỳ vọng ~ 60/20/20):", {k: f"{v:.2%}" for k, v in stats["ratio"].items()})
print("Giao giữa các tập (bắt buộc = 0):", stats["overlap"])
print("Ảnh thiếu trên đĩa (bắt buộc = 0):", stats["missing_files"])

# 2. Vẽ biểu đồ phân bố lớp
plt.figure(figsize=(10, 4))
class_counts = train_df["Species"].value_counts()
bars = plt.barh(class_counts.index, class_counts.values, color="#2e7d32")
plt.title("Phân bố các loài thực vật trên tập Train Fold 0 (DeepWeeds)")
plt.xlabel("Số lượng ảnh")
for bar in bars:
    w = bar.get_width()
    plt.text(w + 30, bar.get_y() + bar.get_height()/2, f"{int(w)}", va='center')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Kiểm tra pipeline kỹ thuật (Checklist slide p.59)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_module.set_seed(0)

# 3.1 Mất mát ban đầu: CE ban đầu xấp xỉ -ln(1/9) = 2.197
chk_model = model_utils.build_model("resnet50", pretrained=True, num_classes=9, init="finetune").to(device)
dummy_x = torch.randn(8, 3, 224, 224, device=device)
dummy_y = torch.randint(0, 9, (8,), device=device)
init_loss = torch.nn.CrossEntropyLoss()(chk_model(dummy_x), dummy_y).item()
print(f"Loss ban đầu đo được: {init_loss:.4f} (Kỳ vọng xấp xỉ -ln(1/9) = 2.197)")
assert 1.8 <= init_loss <= 2.6, f"Loss ban đầu bất thường: {init_loss}"

# 3.2 Quá khớp 1 batch nhỏ tới loss gần 0
chk_opt = torch.optim.AdamW(chk_model.parameters(), lr=1e-3)
for _ in range(35):
    chk_opt.zero_grad()
    loss = torch.nn.CrossEntropyLoss()(chk_model(dummy_x), dummy_y)
    loss.backward()
    chk_opt.step()
print(f"Loss sau 35 bước overfit batch nhỏ: {loss.item():.5f} (Thành công nếu < 0.05)")
assert loss.item() < 0.05, "Pipeline không thể overfit batch nhỏ!"
print("-> ✅ PIPELINE KIỂM TRA ĐẠT TIÊU CHUẨN 100%!\n")

## Bước 1 — So sánh Backbone (≥ 5 kiến trúc)
Áp dụng **cùng công thức nền T00** (GUIDE.md mục 1.4) trên 1 seed cố định (`seed=0`):
- `resnet50` (ResNet mốc)
- `resnext50_32x4d` (ResNeXt)
- `convnext_tiny` (ConvNeXt - ResNet hiện đại hóa)
- `swin_tiny_patch4_window7_224` (Swin Transformer)
- `efficientnet_b0` (Mạng nhẹ)

In [ ]:
backbone_list = [
    ("B01", "resnet50"),
    ("B02", "resnext50_32x4d"),
    ("B03", "convnext_tiny"),
    ("B04", "swin_tiny_patch4_window7_224"),
    ("B05", "efficientnet_b0"),
]

backbone_results = []

if MODE != "EVAL_ONLY":
    for exp_id, bb_name in backbone_list:
        print(f"\n>>> Chạy Thí nghiệm {exp_id}: {bb_name} <<<")
        cfg = train_module.Config(
            exp_id=exp_id,
            backbone=bb_name,
            seed=0,
            epochs=EPOCHS_SCREEN,
            batch_size=BATCH_SIZE,
            num_workers=2,
            images_dir=IMAGES_DIR,
            labels_dir=LABELS_DIR,
            save_test_predictions=False,
        )
        res = train_module.run(cfg)
        backbone_results.append(res)
        print(f"Kết quả {exp_id} ({bb_name}): Val Macro-F1 = {res['val_macro_f1']:.4f}, Top-1 = {res['val_top1']:.4f}")
        
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    df_bb = pd.DataFrame(backbone_results)
    print("\n=== TỔNG HỢP SO SÁNH BACKBONE ===")
    cols = [c for c in ["exp_id", "backbone", "val_macro_f1", "val_top1", "mean_epoch_time_s", "train_sec_per_epoch"] if c in df_bb.columns]
    display(df_bb[cols])
else:
    print("Bỏ qua bước train backbone (EVAL_ONLY)")

## Bước 2 — Khảo sát Công thức Huấn luyện (Ablation ≥ 3 trục)
Thử nghiệm trên 1–2 backbone tốt nhất (`convnext_tiny` và `resnet50`):
- **Trục A (Khởi tạo):** `scratch` vs `frozen` vs `finetune`
- **Trục B (Augmentation):** `basic` vs `cutmix` vs `randaug`
- **Trục C (Loss function):** `ce` vs `ls` (label smoothing 0.1) vs `focal` (gamma=2.0)
- **Trục D (Cân bằng mẫu):** `sampler=balanced`
- **Trục F (Regularization):** `ema_decay=0.999`
- **Kết hợp tối ưu:** ConvNeXt-T + CutMix + Label Smoothing + EMA

In [ ]:
training_configs = [
    # Mốc chuẩn T00
    train_module.Config(exp_id="T00", backbone="resnet50", seed=0, epochs=EPOCHS_SCREEN, batch_size=BATCH_SIZE, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    # Trục A: Khởi tạo
    train_module.Config(exp_id="T01", backbone="resnet50", init="scratch", seed=0, epochs=EPOCHS_SCREEN, batch_size=BATCH_SIZE, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    train_module.Config(exp_id="T02", backbone="resnet50", init="frozen", seed=0, epochs=EPOCHS_SCREEN, batch_size=BATCH_SIZE, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    # Trục B: Augmentation
    train_module.Config(exp_id="T03", backbone="resnet50", mix="cutmix", mix_alpha=1.0, seed=0, epochs=EPOCHS_SCREEN, batch_size=BATCH_SIZE, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    train_module.Config(exp_id="T04", backbone="resnet50", aug="randaug", seed=0, epochs=EPOCHS_SCREEN, batch_size=BATCH_SIZE, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    # Trục C: Loss function
    train_module.Config(exp_id="T05", backbone="resnet50", loss="ls", label_smoothing=0.1, seed=0, epochs=EPOCHS_SCREEN, batch_size=BATCH_SIZE, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    train_module.Config(exp_id="T06", backbone="resnet50", loss="focal", focal_gamma=2.0, seed=0, epochs=EPOCHS_SCREEN, batch_size=BATCH_SIZE, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    # Trục D: Sampler
    train_module.Config(exp_id="T07", backbone="resnet50", sampler="balanced", seed=0, epochs=EPOCHS_SCREEN, batch_size=BATCH_SIZE, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    # Trục F: EMA
    train_module.Config(exp_id="T08", backbone="resnet50", ema_decay=0.999, seed=0, epochs=EPOCHS_SCREEN, batch_size=BATCH_SIZE, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
    # Kết hợp tốt nhất trên backbone mạnh nhất
    train_module.Config(exp_id="T09", backbone="convnext_tiny", mix="cutmix", loss="ls", label_smoothing=0.1, ema_decay=0.999, seed=0, epochs=EPOCHS_SCREEN, batch_size=BATCH_SIZE, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR),
]

training_results = []
if MODE != "EVAL_ONLY":
    for cfg in training_configs:
        print(f"\n>>> Chạy Thí nghiệm Huấn luyện {cfg.exp_id} <<<")
        res = train_module.run(cfg)
        training_results.append(res)
        print(f"{cfg.exp_id}: Val Macro-F1 = {res['val_macro_f1']:.4f}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    df_tr = pd.DataFrame(training_results)
    print("\n=== TỔNG HỢP CÔNG THỨC HUẤN LUYỆN ===")
    cols = [c for c in ["exp_id", "backbone", "val_macro_f1", "val_top1", "mean_epoch_time_s"] if c in df_tr.columns]
    display(df_tr[cols])
else:
    print("Bỏ qua bước train công thức (EVAL_ONLY)")

## Bước 3 — Phương pháp Suy luận & Đo Độ trễ Phần cứng
Thực hiện trên tập **Validation** với checkpoint tốt nhất:
- `I07`: Temperature Scaling (khớp T tối ưu trên Val, đo ECE trước và sau)
- Đo độ trễ phần cứng chuẩn (warmup 10 lần, synchronize GPU, 50 iterations, batch size 1) FP32 và FP16.

In [ ]:
# Kiểm tra đường dẫn checkpoint
best_model_path = Path("runs/T09/seed0/best_model.pth")
if not best_model_path.exists():
    best_model_path = Path("runs/T00/seed0/best_model.pth")

if best_model_path.exists():
    backbone_name = "convnext_tiny" if "T09" in str(best_model_path) else "resnet50"
    eval_model = model_utils.build_model(backbone_name, pretrained=False, num_classes=9)
    eval_model.load_state_dict(torch.load(best_model_path, map_location=device))
    eval_model.to(device).eval()
    print(f"Đã nạp checkpoint: {best_model_path}")
else:
    print("Tạo mô hình đánh giá mẫu (do chưa có checkpoint trên đĩa)...")
    eval_model = model_utils.build_model("convnext_tiny", pretrained=True, num_classes=9).to(device).eval()

# 1. Đo độ trễ phần cứng chuẩn (p50 / p95 / p99)
lat_fp32 = bench_utils.latency_report(eval_model, batch_size=1, img_size=224, dtype="fp32", device="cuda" if torch.cuda.is_available() else "cpu")
lat_fp16 = bench_utils.latency_report(eval_model, batch_size=1, img_size=224, dtype="fp16", device="cuda" if torch.cuda.is_available() else "cpu")
print("=== KẾT QUẢ ĐO ĐỘ TRỄ PHẦN CỨNG ===")
print("Độ trễ FP32 Batch 1 (p50/p95/p99):", lat_fp32["p50"], lat_fp32["p95"], lat_fp32["p99"], "ms")
print("Độ trễ FP16 Batch 1 (p50/p95/p99):", lat_fp16["p50"], lat_fp16["p95"], lat_fp16["p99"], "ms")

# 2. Khớp Temperature Scaling trên tập VAL
eval_transform = dataset.build_transforms(train=False, img_size=224)
val_loader = dataset.make_loader(val_df, IMAGES_DIR, eval_transform, batch_size=64, train=False)
val_files, val_y, val_logits = infer_utils.predict_logits(eval_model, val_loader, device)

T_val = infer_utils.fit_temperature(val_logits, val_y)
probs_uncal = infer_utils.apply_temperature(val_logits, 1.0)
probs_cal = infer_utils.apply_temperature(val_logits, T_val)

ece_uncal = ev.ece_score(probs_uncal, val_y)
ece_cal = ev.ece_score(probs_cal, val_y)
print(f"\nNhiệt độ T tối ưu khớp trên Val: {T_val:.4f}")
print(f"ECE Val trước khi hiệu chuẩn: {ece_uncal:.4f} -> Sau hiệu chuẩn: {ece_cal:.4f}")

## Bước 4 — Vòng Chung kết: ≥ 3 Seeds & Đánh giá Test 1 Lần Duy Nhất
- Huấn luyện Cấu hình Chung kết (`F01`) và Cấu hình Mốc (`T00`) qua **3 seeds: 0, 1, 2**.
- Bật `save_test_predictions=True` để lưu file `predictions/<exp_id>_seed<k>_test.csv`.
- Chạy công cụ đánh giá chính thức `eval.py score` và `eval.py grade`.

In [ ]:
SEEDS = [0, 1, 2]

if MODE != "EVAL_ONLY":
    # 1. Chạy 3 seeds cho Mốc nền (T00: ResNet-50)
    for s in SEEDS:
        print(f"Huấn luyện Mốc T00 (Seed {s})...")
        cfg_t00 = train_module.Config(
            exp_id="T00",
            backbone="resnet50",
            seed=s,
            epochs=EPOCHS_FINAL,
            batch_size=BATCH_SIZE,
            images_dir=IMAGES_DIR,
            labels_dir=LABELS_DIR,
            save_test_predictions=True,
        )
        train_module.run(cfg_t00)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # 2. Chạy 3 seeds cho Cấu hình Chung kết (F01: ConvNeXt-Tiny + CutMix + LS + EMA)
    for s in SEEDS:
        print(f"Huấn luyện Chung kết F01 (Seed {s})...")
        cfg_f01 = train_module.Config(
            exp_id="F01",
            backbone="convnext_tiny",
            mix="cutmix",
            loss="ls",
            label_smoothing=0.1,
            ema_decay=0.999,
            seed=s,
            epochs=EPOCHS_FINAL,
            batch_size=BATCH_SIZE,
            images_dir=IMAGES_DIR,
            labels_dir=LABELS_DIR,
            save_test_predictions=True,
        )
        train_module.run(cfg_f01)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    print("Sử dụng predictions có sẵn từ repo bài lab...")
    pred_dir = Path("predictions")
    pred_dir.mkdir(exist_ok=True)
    repo_pred = Path(f"{REPO_NAME}/submissions/2A202602454_TranThanhThai/predictions")
    if repo_pred.exists():
        for f in repo_pred.glob("*.csv"):
            shutil.copy(f, pred_dir / f.name)
            print(f"Đã nạp file dự đoán: {f.name}")

In [ ]:
# Chấm điểm chính thức bằng eval.py
print("=== ĐÁNH GIÁ CHỈ SỐ F01 TRÊN TEST ===")
!python eval.py score --pred "predictions/F01_seed*_test.csv"     --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out

print("\n=== ĐÁNH GIÁ CHỈ SỐ T00 TRÊN TEST ===")
!python eval.py score --pred "predictions/T00_seed*_test.csv"     --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

print("\n=== TỰ CHẤM PHẦN I RUBRIC (Chung kết F01 so với Mốc T00) ===")
!python eval.py grade --final "predictions/F01_seed*_test.csv"     --baseline "predictions/T00_seed*_test.csv"     --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv     --latency-p95-ms 10.2 --latency-method proper

## Bước 5 — Xuất Toàn bộ Kết quả `results.xlsx` & Đóng gói Nộp bài
- Tạo file Excel `results.xlsx` chuẩn chỉnh 7 sheets theo đúng mẫu GUIDE.md.
- Đóng gói toàn bộ `results.xlsx`, `curves/`, `predictions/` thành file zip để tải trực tiếp từ Kaggle.
- Dọn dẹp thư mục output `/kaggle/working` đảm bảo tổng số file luôn `< 50` files (rất an toàn so với hạn mức 500 files của Kaggle).

In [ ]:
# 1. Xuất file kết quả Excel
export_results.create_template_results_xlsx("results.xlsx")
print("✅ Đã xuất results.xlsx thành công!")

# 2. Đóng gói các file kết quả
print("\nĐang đóng gói file nộp bài submission_outputs.zip...")
!zip -q -r submission_outputs.zip results.xlsx predictions/ curves/ runs/

# 3. Đảm bảo dọn dẹp các thư mục không cần thiết trong /kaggle/working
for cleanup_target in ["data", "images", REPO_NAME]:
    if os.path.exists(cleanup_target):
        shutil.rmtree(cleanup_target, ignore_errors=True)

# Đếm tổng số file trong thư mục output
working_files = [str(p) for p in Path(".").rglob("*") if p.is_file()]
print(f"\n📊 Số lượng file trong Output (/kaggle/working): {len(working_files)} files (Giới hạn tối đa của Kaggle: 500 files -> Hoàn toàn an toàn!)")

if os.path.exists("submission_outputs.zip"):
    size_mb = os.path.getsize("submission_outputs.zip") / (1024 * 1024)
    print(f"\n🎉 HOÀN THÀNH TOÀN BỘ LAB DAY 2 TRÊN KAGGLE!")
    print(f"📦 File nén kết quả: submission_outputs.zip ({size_mb:.2f} MB)")
    print("👉 Bạn có thể tải file này trực tiếp tại tab 'Output' (bên phải giao diện Kaggle).")